# Marine Engine Dual Model Training
Trains a regression model for fuel consumption and a classification model for condition monitoring.


In [ ]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report
from backend.src.preprocessing import EnginePreprocessor, FEATURE_COLUMNS, ENGINE_CONDITIONS

df = pd.read_csv('../data/raw/marine_engine_data.csv')
X = df[FEATURE_COLUMNS]
y_reg = df[['fuel_consumption_kg_h', 'sfoc_g_kwh']]
y_clf = df['condition']

preprocessor = EnginePreprocessor()
X_trans = preprocessor.fit_transform(X)

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X_trans, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)


## 1. Fuel Oil Consumption Regressor


In [ ]:
reg = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
reg.fit(X_train, y_reg_train)
y_reg_pred = reg.predict(X_test)

print(f'Fuel Flow MAE: {mean_absolute_error(y_reg_test.iloc[:, 0], y_reg_pred[:, 0]):.2f} kg/h')
print(f'Fuel Flow R2:  {r2_score(y_reg_test.iloc[:, 0], y_reg_pred[:, 0]):.4f}')
print(f'SFOC MAE:      {mean_absolute_error(y_reg_test.iloc[:, 1], y_reg_pred[:, 1]):.2f} g/kWh')
print(f'SFOC R2:       {r2_score(y_reg_test.iloc[:, 1], y_reg_pred[:, 1]):.4f}')


## 2. Condition Monitoring Classifier


In [ ]:
clf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
clf.fit(X_train, y_clf_train)
y_clf_pred = clf.predict(X_test)

print('Accuracy:', accuracy_score(y_clf_test, y_clf_pred))
print('\nClassification Report:')
print(classification_report(y_clf_test, y_clf_pred, target_names=ENGINE_CONDITIONS))
